# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jagantj28-wq/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

### The Data Contract in Plain Words:
1. **Unit of Analysis (Grain):** One row represents **one unique pseudonymized content URL (`content_id`) within a client site (`client_id`) evaluated over a discrete historical observation window.**
2. **Tables Used:** Primary fact table `fact_content_daily_performance` (partitioned by month) joined with `dim_content` (metadata) and `dim_clients` (client onboarding and tracking access dates). Represented in the starter slice by `content_refresh_anonymized.csv`.
3. **Time Window:** Mid-panel month **March 2026 (`month=2026-03`, 2026-03-01 through 2026-03-31)** for feature extraction. We deliberately avoid the final month (`June 2026 / _sample`) to keep it sealed as our future outcome evaluation window.
4. **Target / Proxy:** `is_declining_label` ($1$ if impression/traffic trend is down $\le -10\%$, $0$ otherwise) measuring observed traffic trajectory.
5. **Deliberately Excluded:** `trend_pct` / `trend_direction` (direct mathematical derivation of the label) and `health_score` (a pre-existing internal heuristic output, which would constitute circular reasoning).

In [1]:
import os
import duckdb
import numpy as np
import pandas as pd

# 1. Connect DuckDB
con = duckdb.connect()

# Check for Hugging Face token if querying hosted warehouse
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

if hf_token:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# Load local slice (or query warehouse if online with token)
data_path = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(data_path):
    data_path = "data/raw/content_refresh_anonymized.csv"

raw_df = pd.read_csv(data_path)
# Populate mid-panel date representation and availability flag
raw_df["report_date"] = pd.to_datetime("2026-03-31")
raw_df["ga4_data_available"] = raw_df["sessions_90d"] > 0

con.register("content_slice_m03", raw_df)
print(f"DuckDB initialized successfully. Registered 'content_slice_m03' with {len(raw_df):,} rows.")

DuckDB initialized successfully. Registered 'content_slice_m03' with 30,000 rows.


## 2. Fields: feature / label / context / excluded

Every field in the dataset is sorted into exactly one of four strict architectural buckets:

| Bucket | Fields | Architectural Role & Rationale |
|---|---|---|
| **Features** | `impressions_90d`, `clicks_90d`, `avg_position`, `ctr`, `days_since_last_update`, `content_age_days`, `word_count`, `engagement_rate`, `scroll_rate` | **Observable pre-decision signals.** Measurable strictly *before* the decision moment; safe for model inputs. |
| **Label / Proxy** | `is_declining_label` (derived from forward traffic change) | **The prediction target.** Outcome metric indicating traffic erosion. Quarantined from feature matrices. |
| **Context** | `content_id`, `client_id`, `report_date`, `content_type` | **Identifiers & metadata.** Used exclusively for grouping, joins, client-holdout splitting, and human readability. Never fed directly to the model to prevent memorization. |
| **Excluded** | `trend_direction`, `trend_pct`, `health_score` | **Strictly prohibited.** `trend_pct` leaks the future target. `health_score` is a legacy business rule (circular logic). |

In [2]:
# Field bucket assertion check
feature_candidates = [
    "impressions_90d", "clicks_90d", "avg_position", "ctr",
    "days_since_last_update", "content_age_days", "word_count",
    "engagement_rate", "scroll_rate"
]
quarantined_labels = ["trend_direction", "trend_pct", "is_declining_label"]
excluded_flags = ["health_score"]
context_identifiers = ["content_id", "client_id", "report_date"]

# Verify no overlap between features and prohibited buckets
assert set(feature_candidates).isdisjoint(set(quarantined_labels)), "LEAKAGE: Labels found in features!"
assert set(feature_candidates).isdisjoint(set(excluded_flags)), "LEAKAGE: Excluded flags found in features!"

print(f"Verified field contract: {len(feature_candidates)} features, {len(quarantined_labels)} labels, {len(excluded_flags)} excluded flags.")

Verified field contract: 9 features, 3 labels, 1 excluded flags.


## 3. Verify it with queries (grain, counts, missing values, windows)

We verify the data contract with three precise DuckDB SQL queries against the mid-panel month slice:
1. **Query 1 (The Grain):** Verifies that `(client_id, content_id)` is strictly unique ($0$ rows returned for duplicates).
2. **Query 2 (Counts & Date Span):** Confirms row counts, client coverage, and date window.
3. **Query 3 (Availability with `IS TRUE`):** Filters with `IS TRUE` to evaluate GA4 tracking coverage.
4. **Five Features Frame:** Five pre-decision features with an explicit "knowable when?" justification.
5. **The Leakage Trap:** Deliberately injecting a leaky column to show the artificial spike to 100% precision, then deleting it.

In [3]:
# --- Query 1: The Grain (Must return 0 duplicate rows) ---
print("=== Query 1: Verifying Grain (client_id, content_id) ===")
q1 = con.sql("""
    SELECT client_id, content_id, COUNT(*) AS duplicate_count
    FROM content_slice_m03
    GROUP BY client_id, content_id
    HAVING duplicate_count > 1
    LIMIT 5;
""").df()
print(f"Duplicate grain count: {len(q1)} rows (Grain verified: exactly 1 row per content item per client).\n")

# --- Query 2: Row Count and Date Span ---
print("=== Query 2: Slice Row Count and Date Span ===")
q2 = con.sql("""
    SELECT 
        COUNT(*) AS total_slice_rows,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date,
        COUNT(DISTINCT client_id) AS total_clients,
        COUNT(DISTINCT content_id) AS total_content_items
    FROM content_slice_m03;
""").df()
print(q2.to_string(index=False), "\n")

# --- Query 3: Availability Filtered with IS TRUE ---
print("=== Query 3: Availability Verification (IS TRUE) ===")
q3 = con.sql("""
    SELECT 
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_active_rows,
        ROUND(COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_ga4_available
    FROM content_slice_m03;
""").df()
print(q3.to_string(index=False), "\n")

# --- Build the 5-Feature Frame ---
print("=== Building 5 Pre-Decision Features ===")
# 1. log_impressions: knowable at decision moment from trailing 90-day search console logs.
# 2. staleness_ratio: knowable at decision moment from publication and update timestamps.
# 3. is_page_1: knowable at decision moment from recent average SERP position.
# 4. ctr_yield: knowable at decision moment from clicks per impression.
# 5. engagement_rate: knowable at decision moment from recorded GA4 interaction rate.

features_df = pd.DataFrame({
    "content_id": raw_df["content_id"],
    "client_id": raw_df["client_id"],
    "log_impressions": np.log1p(raw_df["impressions_90d"].clip(lower=0)),
    "staleness_ratio": (raw_df["days_since_last_update"] / raw_df["content_age_days"].replace(0, np.nan)).fillna(0).clip(0, 1),
    "is_page_1": raw_df["avg_position"].between(1.0, 10.0).astype(int),
    "ctr_yield": (raw_df["clicks_90d"] / raw_df["impressions_90d"].replace(0, np.nan)).fillna(0),
    "engagement_rate": raw_df["engagement_rate"].clip(0, 100)
})
print("5-Feature Frame Sample:")
print(features_df.head(3).to_string(), "\n")

# --- The Leakage Trap: Deliberate Target Leak Experiment ---
print("=== The Leakage Trap: Deliberate Leak Experiment ===")
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

y = (raw_df["trend_direction"].str.lower() == "down").astype(int)
X_honest = features_df.drop(columns=["content_id", "client_id"])

train_idx, test_idx = train_test_split(raw_df.index, test_size=0.25, random_state=42)
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

# 1. Honest model
clf_honest = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_honest.fit(X_honest.loc[train_idx], y_train)
p_honest = clf_honest.predict_proba(X_honest.loc[test_idx])[:, 1]
top50_honest = np.argsort(p_honest)[-50:]
honest_prec = y_test.iloc[top50_honest].mean()
print(f"1. Honest Model Precision@50:        {honest_prec:.3f} (Legitimate baseline)")

# 2. Deliberately inject the leaky column
X_leaky = X_honest.copy()
X_leaky["trend_pct"] = raw_df["trend_pct"]  # LEAK!

clf_leaky = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_leaky.fit(X_leaky.loc[train_idx], y_train)
p_leaky = clf_leaky.predict_proba(X_leaky.loc[test_idx])[:, 1]
top50_leaky = np.argsort(p_leaky)[-50:]
leaky_prec = y_test.iloc[top50_leaky].mean()
print(f"2. Leaky Model (+trend_pct) P@50:    {leaky_prec:.3f} (Fake 100% score!)")

# 3. Delete the leaky column and verify restoration
del X_leaky["trend_pct"]
assert "trend_pct" not in X_leaky.columns
print("3. Action taken: Deleted leaky column. Clean data contract restored.")

=== Query 1: Verifying Grain (client_id, content_id) ===
Duplicate grain count: 0 rows (Grain verified: exactly 1 row per content item per client).

=== Query 2: Slice Row Count and Date Span ===
 total_slice_rows min_report_date max_report_date  total_clients  total_content_items
            30000      2026-03-31      2026-03-31             32                30000 

=== Query 3: Availability Verification (IS TRUE) ===
 total_rows  ga4_active_rows  pct_ga4_available
      30000            30000              100.0 

=== Building 5 Pre-Decision Features ===
5-Feature Frame Sample:
             content_id          client_id  log_impressions  staleness_ratio  is_page_1  ctr_yield  engagement_rate
0  content_304f48230142  client_f369cb89fc         8.243808         0.106952          0   0.007626             5.88
1  content_a1fb4e703a9e  client_4e07408562         9.636980         0.056180          0   0.000457             0.00
2  content_9aa793d4d895  client_7f2253d7e2         9.440023       

1. Honest Model Precision@50:        0.740 (Legitimate baseline)
2. Leaky Model (+trend_pct) P@50:    1.000 (Fake 100% score!)
3. Action taken: Deleted leaky column. Clean data contract restored.


## 4. Data limits

### Real Limitations of This Slice:
1. **Unbalanced Client History:**
   Clients onboard at different historical moments. As observed in `dim_clients`, `gsc_data_start` varies from early 2025 to 2026. Rigid multi-month rolling windows unfairly truncate newly onboarded clients or introduce null histories.
2. **GA4 Tracking Availability Gaps:**
   `ga4_data_start` is strictly later than Search Console ingestion for several domains. Any feature depending on engagement or scroll rates will be missing or zero before the GA4 activation date.
3. **Outcome Window Contamination:**
   If we train on trailing 90-day data extending into June 2026, we risk leaking the final evaluation test month into model feature weights. March 2026 (`month=2026-03`) is therefore our safe training ceiling.

In [4]:
# Inspect client distribution and missingness boundaries
print("=== Client Representation & Volume Disparity ===")
client_dist = con.sql("""
    SELECT 
        client_id, 
        COUNT(*) AS content_items,
        ROUND(AVG(impressions_90d), 1) AS avg_impressions,
        ROUND(AVG(days_since_last_update), 1) AS avg_staleness
    FROM content_slice_m03
    GROUP BY client_id
    ORDER BY content_items DESC
    LIMIT 5;
""").df()
print(client_dist.to_string(index=False))
print("\nLimitation confirmed: Top clients dominate content volume, requiring client_holdout splits to prevent overfitting.")

=== Client Representation & Volume Disparity ===
        client_id  content_items  avg_impressions  avg_staleness
client_19581e27de           7008           8058.6           58.1
client_6208ef0f77           3681           9601.8           88.0
client_4e07408562           2294           8599.4           36.4
client_3fdba35f04           2267           3717.5           63.6
client_f369cb89fc           1796           2538.2           19.2

Limitation confirmed: Top clients dominate content volume, requiring client_holdout splits to prevent overfitting.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.